# Enhanced Summarization Pipeline


In [5]:
!pip install rapidfuzz spacy pandas numpy langdetect

In [6]:
!pip freeze > requirements.txt

In [1]:
import sys
sys.path.insert(0, '/home/astronaut/sumitup-ai-meet/backend')
from pprint import pprint
import asyncio
from motor.motor_asyncio import AsyncIOMotorClient
from beanie import init_beanie
from models.models import Transcripts, User, Tenant, ActionItems, Meeting, Participants, Team, Billing
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

async def fetch_transcripts():
    # Initialize MongoDB connection
    MONGO_URI = os.getenv("MONGO_URI", "mongodb://127.0.0.1:27017")
    DB_NAME = os.getenv("DB_NAME", "test")
    
    client = AsyncIOMotorClient(MONGO_URI)
    db = client[DB_NAME]
    
    # Initialize Beanie
    await init_beanie(db, document_models=[User, Tenant, ActionItems, Meeting, Participants, Transcripts, Team, Billing])
    
    # Now fetch transcripts
    transcripts = await Transcripts.find().to_list()
    return transcripts

# Run the async function
transcripts = await fetch_transcripts()
all_transcripts_data = [{
    "transcript": t.transcript,
    "speaker_name": t.speaker_name,
    "speaker_id": t.speaker_id,
    "timestamp_ms": t.timestamp_ms,
    "duration_ms": t.duration_ms,
} for t in transcripts] 

pprint(all_transcripts_data)

[{'duration_ms': 39490,
  'speaker_id': 'spaces/8-nvQrg1dsIB/devices/352',
  'speaker_name': 'Alhaan Ahmed',
  'timestamp_ms': 1775753623852,
  'transcript': 'Are you listening my voice today? Alright. Let me introduce '
                'you myself. I am CEO, cofounder of Samika. Right? What is '
                'Samika? It is a soft software service, SaaS platform, '
                'designed to automate the business. What it does, it joins '
                "everything's net, complete the transcription, and converts "
                'into structured summary. The structured summary is important '
                'because it converts all the discussion details into a '
                'structured format so that you can see and remember what was '
                'said and what is what Thank you. What is moment time'},
 {'duration_ms': 127840,
  'speaker_id': 'spaces/8-nvQrg1dsIB/devices/352',
  'speaker_name': 'Alhaan Ahmed',
  'timestamp_ms': 1775753668028,
  'transcript': 'I hope, t

In [2]:
import pandas as pd

df = pd.DataFrame(all_transcripts_data)

df


,transcript,speaker_name,speaker_id,timestamp_ms,duration_ms
0,Are you listening my voice today? Alright. Let...,Alhaan Ahmed,spaces/8-nvQrg1dsIB/devices/352,1775753623852,39490
1,"I hope, this is something I need to remember a...",Alhaan Ahmed,spaces/8-nvQrg1dsIB/devices/352,1775753668028,127840
2,"I hope we have, cleared our the agenda or meet...",Alhaan Ahmed,spaces/8-nvQrg1dsIB/devices/352,1775753862407,9700
3,Transcribe your meetings. That joins your meet...,Alhaan Ahmed,spaces/IKsYUwUg2RQB/devices/480,1776162240317,68570
4,"If you have any questions, you can ask me. Or ...",Alhaan Ahmed,spaces/IKsYUwUg2RQB/devices/480,1776162381983,9030
...,...,...,...,...,...
57,"Okay, students. Now you will have to show me y...",Alhaan Ahmed,16778240,1776923786395,42650
58,"Yeah. Join as a part, in your platforms like G...",Alhaan Ahmed,spaces/HhbMdIS6JDAB/devices/390,1777290308486,103380
59,A maior.,Danish Ansari,16782336,1777543940995,4340
60,"The ex expected time to complete this is, in t...",Danish Ansari,16782336,1777544000388,11140


In [3]:
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

DetectorFactory.seed = 0

def filter_by_language(transcripts: list) -> dict:
    results = []
    for t in transcripts:
        try:
            language = detect(t["transcript"])
            is_mixed_or_foreign = language != "en"
            recommended_model = recommended_model = "llama-3.3-70b-versatile" if is_mixed_or_foreign else "llama-3.1-8b-instant"
            transcript_segment_length = len(t["transcript"])

            data = {
                "language": language,
                "transcripts": t["transcript"],
                "selected_model": recommended_model,
                "total_segments": transcript_segment_length,
            }
            results.append(data)
        except LangDetectException:
            print("Failed to Detect")

    
    return results

filtered_transcripts = filter_by_language(all_transcripts_data)
print("Language Info: ")
pprint(filtered_transcripts)

    

Language Info: 
[{'language': 'en',
  'selected_model': 'llama-3.1-8b-instant',
  'total_segments': 516,
  'transcripts': 'Are you listening my voice today? Alright. Let me introduce '
                 'you myself. I am CEO, cofounder of Samika. Right? What is '
                 'Samika? It is a soft software service, SaaS platform, '
                 'designed to automate the business. What it does, it joins '
                 "everything's net, complete the transcription, and converts "
                 'into structured summary. The structured summary is important '
                 'because it converts all the discussion details into a '
                 'structured format so that you can see and remember what was '
                 'said and what is what Thank you. What is moment time'},
 {'language': 'en',
  'selected_model': 'llama-3.1-8b-instant',
  'total_segments': 1750,
  'transcripts': 'I hope, this is something I need to remember about this '
                 'particular th

In [10]:
df2 = pd.DataFrame(filtered_transcripts)

df2

,language,transcripts,selected_model,total_segments
0,en,Are you listening my voice today? Alright. Let...,llama-3.1-8b-instant,516
1,en,"I hope, this is something I need to remember a...",llama-3.1-8b-instant,1750
2,en,"I hope we have, cleared our the agenda or meet...",llama-3.1-8b-instant,145
3,en,Transcribe your meetings. That joins your meet...,llama-3.1-8b-instant,826
4,en,"If you have any questions, you can ask me. Or ...",llama-3.1-8b-instant,101
...,...,...,...,...
57,en,"Okay, students. Now you will have to show me y...",llama-3.1-8b-instant,546
58,en,"Yeah. Join as a part, in your platforms like G...",llama-3.1-8b-instant,1273
59,pt,A maior.,llama-3.3-70b-versatile,8
60,en,"The ex expected time to complete this is, in t...",llama-3.1-8b-instant,121


**Pipeline:**
> Step 1 -> Detect Base - Language (Already Done) 

> Step 2 -> Remove Duplicate words such as "the the", "Ah", "uhm"

In [5]:
import re
import spacy
from rapidfuzz import fuzz

# Loading with minimal components for speed
nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer", "attribute_ruler"])

def remove_deduplication_and_cleaning(filtered_transcripts: list):
    # 1. Combine English text
    combined_text = " ".join([t["transcripts"] for t in filtered_transcripts if t["language"] == "en"])
    if not combined_text:
        return "", 0
    old_count = len(combined_text.split(" "))
    # 2. Basic cleaning (whitespace)
    text = re.sub(r'\s+', ' ', combined_text).strip()
    
    # 3. Remove consecutive repeating phrases (e.g., "look after look after")
    # This regex looks for a group of words followed by the exact same group
    # \b is a word boundary, \w+ is one or more words, \1 is the backreference
    text = re.sub(r'\b(\w+(?:\s+\w+)*)\s+\1\b', r'\1', text, flags=re.IGNORECASE)

    # 4. Remove Filler Words (Linguistic Cleaning)
    FILLER_WORDS = {"um", "uh", "uhm", "ah", "like", "actually", "basically", "you know", "i mean", "right"}
    
    doc = nlp(text)
    cleaned_sentences = []
    
    for sent in doc.sents:
        # Filter fillers
        filtered_tokens = [
            t.text for t in sent 
            if t.text.lower() not in FILLER_WORDS and t.pos_ != "INTJ" and not t.is_punct
        ]
        
        current_s = " ".join(filtered_tokens).strip()
        if not current_s: continue

        # 5. Semantic Sentence Deduplication (RapidFuzz)
        if cleaned_sentences:
            if fuzz.ratio(current_s.lower(), cleaned_sentences[-1].lower()) > 85:
                continue
        
        cleaned_sentences.append(current_s)

    final_text = " ".join(cleaned_sentences)
    
    # 6. Count words
    word_count = len(final_text.split())
    
    return final_text, word_count, old_count

# Usage example:
# cleaned_text, count = remove_deduplication_and_cleaning(filtered_transcripts)
# print(f"Cleaned Transcript Word Count: {count}")

cleaned_sentences, count, oc = remove_deduplication_and_cleaning(filtered_transcripts=filtered_transcripts)
print(cleaned_sentences)
# print(count)
# print(oc)

Are you listening my voice today Alright Let me introduce you myself I am CEO cofounder of Samika What is Samika It is a soft software service SaaS platform designed to automate the business What it does it joins everything 's net complete the transcription and converts into structured summary The structured summary is important because it converts all the discussion details into a structured format so that you can see and remember what was said and what is what Thank you What is moment time I hope this is something I need to remember about this particular thing And I know that this is just something I do n't about But somewhere there is something which is I think it 's a good it 's a very good thing to be happy in the future And Yeah Other tools Otter dot ai fireflies.ai Crisp and other tools which we use as specific something with different things and we have been talking different differentiators For example if we talk about other meeting assistance and other meeting assistant so wh

In [15]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 MB 660.4 kB/s eta 0:00:0000:0100:07
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 1.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.3/341.3 kB 714.7 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 527.8 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 640.4 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 930.8/930.8 kB 2.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 635.7 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 1.2 MB/s eta 0:00:00a 0:00:01m
  Attempting uninstall: setuptools
    Fou

In [16]:
!pip install transformers sentencepiece tiktoken


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 620.0 kB/s eta 0:00:0000:0100:01


In [23]:
!pip install sentencepiece protobuf

  Using cached sentencepiece-0.2.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (10 kB)
Using cached sentencepiece-0.2.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (1.4 MB)


In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Buntan/gec-t5-v1_1-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Save to a folder named 'local_model'
tokenizer.save_pretrained("./local_model")
model.save_pretrained("./local_model")


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 913.44it/s] 
Exception ignored in: <function WeakKeyDictionary.__init__.<locals>.remove at 0x7a88c8fa0220>
Traceback (most recent call last):
  File "/usr/lib/python3.12/weakref.py", line 369, in remove
    def remove(k, selfref=ref(self)):

KeyboardInterrupt: 
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Writing model shards: 100%|██████████| 1/1 [00:06<00:00,  6.47s/it]


In [8]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_path = "./local_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# Test with different prefixes
test_cases = [
    ("This matter will by be reviewed", "No prefix"),
    ("grammar: This matter will by be reviewed", "grammar: prefix"),
    ("fix grammar: This matter will by be reviewed", "fix grammar: prefix"),
    ("gec: This matter will by be reviewed", "gec: prefix"),
    ("correct: This matter will by be reviewed", "correct: prefix"),
]

for text, prefix_type in test_cases:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model.generate(inputs["input_ids"], max_length=128, num_beams=4)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"{prefix_type}:")
    print(f"  Input: {text[:50]}...")
    print(f"  Output: {result[:50]}...")
    print()

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 644.77it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


No prefix:
  Input: This matter will by be reviewed...
  Output: This matter will by be reviewed...

grammar: prefix:
  Input: grammar: This matter will by be reviewed...
  Output: grammar: This matter will be reviewed...

fix grammar: prefix:
  Input: fix grammar: This matter will by be reviewed...
  Output: fix grammar: This matter will by be reviewed...

gec: prefix:
  Input: gec: This matter will by be reviewed...
  Output: gec: This matter will by be reviewed...

correct: prefix:
  Input: correct: This matter will by be reviewed...
  Output: correct: This matter will by be reviewed...



In [16]:
import torch

def correct_text(text):
    input_text = f"grammar: {text}"
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model.generate(inputs["input_ids"],
                                  max_length=512,
                                  num_beams=4,
                                  early_stopping=True,
                                  temperature=0.1
                                  )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test it
test_cases = (
    "Are you listening my voice today Alright Let me introduce you myself I am CEO cofounder of Samika What is Samika It is a soft software service SaaS platform designed to automate the business",
    "And I know that this is just something I do n't about But somewhere there is something which is I think it 's a good it 's a very good thing to be happy in the future And Yeah Other tools Otter dot ai fireflies.ai Crisp",
    "That 's a word but unfortunately the problem you are using again maybe there was an Internet issue That 's why it 's happening I have to discuss it How much we have achieved so far",
    "I want to see how you have divided the task amongst yourself how your work works how some logo is being placed on the dashboard everything Kindly tell me Okay The Sumitup dash the Sumitup logo is placed on the dashboard",
    "I speak here is transcribed and it is converted into raw summary which we can visualize here which you can see or so what it does it aligns it helps the the leader the manager or the head of the team"
)
for test in test_cases:
    print(correct_text(test))



def calculate_accuracy():
    """Calculate accuracy of grammar correction on test cases."""
    
    test_cases = [
        "Are you listening my voice today Alright Let me introduce you myself I am CEO cofounder of Samika What is Samika It is a soft software service SaaS platform designed to automate the business",
        "And I know that this is just something I do n't about But somewhere there is something which is I think it 's a good it 's a very good thing to be happy in the future And Yeah Other tools Otter dot ai fireflies.ai Crisp",
        "That 's a word but unfortunately the problem you are using again maybe there was an Internet issue That 's why it 's happening I have to discuss it How much we have achieved so far",
        "I want to see how you have divided the task amongst yourself how your work works how some logo is being placed on the dashboard everything Kindly tell me Okay The Sumitup dash the Sumitup logo is placed on the dashboard",
        "I speak here is transcribed and it is converted into raw summary which we can visualize here which you can see or so what it does it aligns it helps the the leader the manager or the head of the team"
    ]
    
    print("=" * 70)
    print("GRAMMAR CORRECTION ACCURACY REPORT")
    print("=" * 70)
    
    results = []
    for i, test in enumerate(test_cases, 1):
        corrected_raw = correct_text(test)
        
        # 🔥 FIX: Strip the "grammar: " prefix
        if corrected_raw.startswith("grammar: "):
            corrected = corrected_raw[9:]  # Remove "grammar: "
        else:
            corrected = corrected_raw
        
        # Calculate simple metrics
        original_words = len(test.split())
        corrected_words = len(corrected.split())
        
        # Check for improvements
        has_period = '.' in corrected
        has_capital = corrected[0].isupper() if corrected else False
        word_change = abs(original_words - corrected_words)
        
        results.append({
            "test": i,
            "original": test[:50] + "...",
            "corrected": corrected[:80] + "..." if len(corrected) > 80 else corrected,
            "periods": "✅" if has_period else "❌",
            "capitalized": "✅" if has_capital else "❌",
            "word_change": word_change
        })
        
        print(f"\nTest {i}:")
        print(f"  Original: {test[:80]}...")
        print(f"  Corrected: {corrected[:80]}...")
        print(f"  Periods: {has_period} | Capitalized: {has_capital} | Words changed: {word_change}")
    
    # Summary
    print("\n" + "=" * 70)
    print("SUMMARY")
    print("=" * 70)
    
    periods_count = sum(1 for r in results if r["periods"] == "✅")
    caps_count = sum(1 for r in results if r["capitalized"] == "✅")
    
    print(f"✓ Periods added: {periods_count}/{len(test_cases)} ({periods_count/len(test_cases)*100:.0f}%)")
    print(f"✓ Capitalization fixed: {caps_count}/{len(test_cases)} ({caps_count/len(test_cases)*100:.0f}%)")
    print(f"✓ Average words changed: {sum(r['word_change'] for r in results)/len(results):.1f}")
    print("=" * 70)

# Run the accuracy check
calculate_accuracy()
# Run the accuracy check
# Expected output: "This matter will be reviewed by the team"

grammar: Are you listening to my voice today? Right Let me introduce you myself. I am the CEO cofounder of Samika. It is a soft software service SaaS platform designed to automate the business
grammar: And I know that this is just something I don't about But somewhere there is something which I think it's a good it's a very good thing to be happy in the future And Yeah Other tools Otter dot ai fireflies.ai Crisp
grammar: That's a word but unfortunately the problem you are using again maybe there was an Internet issue. That's why it's happening. I have to discuss it How much we have achieved so far.
grammar: I want to see how you have divided the task amongst yourself how your work works how some logo is being placed on the dashboard everything Kindly tell me OK The Sumitup dash the Sumitup logo is placed on the dashboard.
grammar: I speak here is transcribed and it is converted into raw summary which we can visualize here which you can see or so what it does it aligns it helps the lead

This demonstrates that it performs when english spoken is fluent and correct and may auto correct words, however these kind of **garbage words** cannot be fixed or model underperforms under these conditions

In [ ]:
# Final Text Preprocessing Pipeline used for cleaning filler words and then grammar correction

class TranscriptionPreProcessing:
    def __init__(self, transcript_chunk):
        self.transcript_chunk = transcript_chunk

# Refine Method - Best Short Meetings
# Map - Reduce Method - Best for Long Meetings